# Bi-weekly Cross-Sectional Signal Forecasting — Season 1

This notebook walks through the data, builds a predictor, and — most importantly —
shows the two things that decide your score in this competition and did **not**
exist in earlier ones:

1. **The target is not disclosed.** It is constructed so that the obvious first
   signals score ~0 — the demo below shows this happening to a textbook strategy.
2. **Turnover costs 5 bp.** Position churn is charged on every rebalance and drags
   Sharpe by more than most edges on this target are worth.

Read `COMPETITION.md` for the rules; this notebook is the hands-on tour.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from data_loader import list_periods, load_period
from evaluation import backtest, sharpe_ratio, returns_to_equity
from predictor import Predictor
from walkforward import walkforward_backtest

warnings.filterwarnings('ignore')

DATA_DIR = 'data/train'
UNIVERSE = 'small'

## 1. The data

Period files are **cumulative** — period `NNN` already contains all history up to
that point, so you load one file, not a concatenation.

Ticker identities are **not** stable across periods: `ticker.1` in period 001 is a
different blend of assets than `ticker.1` in period 002. Cross-sectional features
transfer; per-ticker habits do not.

In [ ]:
periods = list_periods(DATA_DIR, UNIVERSE)
print(f'Periods available: {len(periods)}  ({periods[0]} .. {periods[-1]})')

data = load_period(periods[-1], DATA_DIR, UNIVERSE)
features, returns, target = data['features'], data['returns'], data['target']
print(f"features {features.shape}   returns {returns.shape}   target {target.shape}")

In [ ]:
# 6 features x 20 tickers, MultiIndex columns (feature, ticker).
# What each column measures, and how it is computed, is not disclosed.
features.columns.get_level_values(0).unique().tolist()


In [ ]:
features.head(3)

### What `returns` actually is

**Not** a raw price return: it is the competition's own return series, and how it is
constructed is deliberately not disclosed. Two consequences you can see directly in
the data below.

First, the earliest rows of a panel can be exactly `0.0` — the construction needs a
warm-up window of history before its first value. That is expected, not corruption.

In [ ]:
warm = (returns.abs().sum(axis=1) == 0).sum()
print(f'rows that are exactly zero (warm-up): {warm}')
returns.iloc[warm - 2:warm + 3].iloc[:, :4]

Second — and this is the heart of the competition — **the first textbook signal
most contestants would build earns nothing here**. Below we construct a completely
standard cross-sectional strategy from a shipped feature and score it. On a
raw-return target it would be a reasonable baseline; on this competition's target
its Sharpe sits at approximately zero, because the target is constructed so that
signals like it are not rewarded. We are not telling you why — working that out is
part of the game — only that the obvious first ideas are unpaid work here.


In [ ]:
def xs_demean(df):
    return df.sub(df.mean(axis=1), axis=0)

# a completely standard cross-sectional strategy built from one shipped feature
textbook = xs_demean(features['Feature.1'].rolling(120, min_periods=1).mean())

print(f'textbook signal, Sharpe on the contest target: {sharpe_ratio(textbook, returns):+.4f}')


## 2. The cost lever

Your score is net of a 5 bp charge on turnover. It is easy to build a signal whose
gross edge is entirely consumed by rebalancing — so always look at gross and net
side by side.

The cell below takes one signal and smooths it over time with an exponentially
weighted mean (causal — only past rows). Watch what happens to the cost column.

In [ ]:
def gross_and_net(signal, rets):
    gross = (signal.shift(1) * rets).sum(axis=1)
    turnover = signal.shift(1).fillna(0.0).diff().abs().sum(axis=1)
    cost = 5e-4 * turnover
    net = gross - cost
    return pd.Series({
        'gross Sharpe': gross.mean() / gross.std(),
        'net Sharpe': net.mean() / net.std(),
        'cost / gross vol': cost.mean() / gross.std(),
    })

raw_signal = xs_demean(features['Feature.2'])
comparison = pd.DataFrame({
    'no smoothing': gross_and_net(raw_signal, returns),
    'EWM span 48': gross_and_net(xs_demean(raw_signal.ewm(span=48).mean()), returns),
    'EWM span 240': gross_and_net(xs_demean(raw_signal.ewm(span=240).mean()), returns),
})
comparison.round(4)

The unsmoothed signal pays a cost worth roughly 6% of its own PnL volatility — a
Sharpe drag of about 0.06, which is larger than most edges you will find on this
target. Smoothing cuts that to ~1% while keeping most of the information. Tune it
early, not last: on the shipped example this single change is worth more than every
feature adjustment combined.

## 3. Write a predictor

Your submission is a `Predictor` subclass. Everything must live **inside the class**
(imports at module level are fine), and `predict()` must return cross-sectionally
de-meaned predictions.

In [ ]:
class MyPredictor(Predictor):
    """Minimal, honest starting point: rank-based features + turnover control."""

    SMOOTH = 240

    def _engineer(self, features):
        f1 = features['Feature.1']
        n = f1.shape[1]

        def csrank(df):
            r = df.rank(axis=1, method='average')
            return (r - 0.5 * (n + 1)) / (0.5 * (n - 1))

        return csrank(f1.rolling(120, min_periods=1).mean())

    def train(self, features, target):
        # A fitted model would go here; this baseline is deliberately fit-free
        # so the mechanics stay visible.
        self.trained = True

    def predict(self, features):
        signal = self._engineer(features)
        signal = signal.ewm(span=self.SMOOTH, min_periods=1).mean()
        return signal.sub(signal.mean(axis=1), axis=0).fillna(0.0)

## 4. Validate

`walkforward_backtest` holds out the tail of the latest period; the `--full` mode of
`runner.py` replays every period and is the number to iterate on.

In [ ]:
results = walkforward_backtest(
    MyPredictor, data_dir=DATA_DIR, universe=UNIVERSE, verbose=True,
)
print(f"\nValidation Sharpe: {results['sharpe']:+.4f}")
print(f"De-mean check: {'PASS' if results['demean_check'] else 'FAIL'}")

In [ ]:
equity = returns_to_equity(results['pf_returns'])
equity.plot(title='Validation equity curve (net of costs)', figsize=(9, 3.5));

## 5. Before you submit

**This notebook is not a submission.** Everything above — loading a period,
printing shapes, the cost comparison, the equity plot — runs at module level,
which `CODE_OUTSIDE_CLASS` rejects. That is deliberate: exploration belongs here,
not in the file you send. What you submit is the class from section 3 and nothing
else at module level — `demo.py` is that shape, worked end to end. If you submit a
notebook, it must hold the same thing: imports, the class, and any scratch cell
wrapped in `if __name__ == "__main__":`, which the scorer never runs.

```bash
python runner.py your_submission.py             # quick check
python runner.py your_submission.py --full      # full walk-forward — iterate on this
python runner.py your_submission.py --gauge-fix # + city / novelty vs existing signals
```

Checklist:

- everything inside the class, no top-level helpers;
- predictions de-meaned (each row sums to zero);
- no future-looking operations (`shift(-n)`, `bfill`, centred windows);
- `train()` under 4 minutes, `predict()` under 60 seconds, no GPU;
- extra dependencies declared with PEP 723 at the top of your `.py`.

Two expectations worth setting: your local numbers will differ from the leaderboard
(part of the scored history is deliberately withheld from the package), and the
overfitting gate rejects signals that cannot beat their own shuffled null — including
honest but weak ones. `COMPETITION.md` explains both.